### Procesamiento de Lenguaje Natural I
# **Desafío 1**



### Vectorización de texto y modelo de clasificación Naïve Bayes con el dataset 20 newsgroups

In [3]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import f1_score

Utilizamos **20newsgroups** por ser un dataset clásico de NLP ya viene incluido y formateado en sklearn

In [4]:
from sklearn.datasets import fetch_20newsgroups
import numpy as np

## Carga de datos

Cargamos los datos (ya separados de forma predeterminada en train y test)

El dataset 20 Newsgroups contiene aproximadamente 18 000 publicaciones de grupos de noticias distribuidas en 20 temas. Está dividido en dos subconjuntos: uno para entrenamiento (train set) y otro para pruebas (test set).

In [5]:
newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
newsgroups_test = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'))

## Vectorización

Instanciamos un vectorizador.

Podemos ver diferentes parámetros de instanciación en la documentación de sklearn https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html

In [6]:
tfidfvect = TfidfVectorizer()

En el atributo `data` accedemos al texto

In [7]:
print(newsgroups_train.data[0])

I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
have on this funky looking car, please e-mail.


Con la interfaz habitual de sklearn podemos ajustar el vectorizador (obtener el vocabulario y calcular el vector IDF) y transformar directamente los datos.

Podemos denominar `X_train` como la matriz documento-término.

In [8]:
X_train = tfidfvect.fit_transform(newsgroups_train.data)

Recordemos que las vectorizaciones por conteos son de tipo sparse, por ello sklearn convenientemente devuelve los vectores de documentos como matrices de tipo sparse.

In [9]:
print(type(X_train))
print(f'shape: {X_train.shape}')
print(f'Cantidad de documentos: {X_train.shape[0]}')
print(f'Tamaño del vocabulario (dimensionalidad de los vectores): {X_train.shape[1]}')

<class 'scipy.sparse._csr.csr_matrix'>
shape: (11314, 101631)
Cantidad de documentos: 11314
Tamaño del vocabulario (dimensionalidad de los vectores): 101631


Una vez ajustado el vectorizador, podemos acceder a atributos como el vocabulario aprendido. Es un diccionario que va de términos a índices.

El índice es la posición en el vector de documento.

In [10]:
tfidfvect.vocabulary_['car']

25775

Probamos con una palbra que no está en el documento.

In [11]:
tfidfvect.vocabulary_['cocoliso']

KeyError: 'cocoliso'

Es muy útil tener el diccionario opuesto que va de índices a términos

In [12]:
idx2word = {v: k for k,v in tfidfvect.vocabulary_.items()}

En `y_train` guardamos los targets que son enteros

In [13]:
y_train = newsgroups_train.target
y_train[:10]

array([ 7,  4,  4,  1, 14, 16, 13,  3,  2,  4])

Hay 20 clases correspondientes a los 20 grupos de noticias

In [14]:
print(f'clases {np.unique(newsgroups_test.target)}')
newsgroups_test.target_names

clases [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]


['alt.atheism',
 'comp.graphics',
 'comp.os.ms-windows.misc',
 'comp.sys.ibm.pc.hardware',
 'comp.sys.mac.hardware',
 'comp.windows.x',
 'misc.forsale',
 'rec.autos',
 'rec.motorcycles',
 'rec.sport.baseball',
 'rec.sport.hockey',
 'sci.crypt',
 'sci.electronics',
 'sci.med',
 'sci.space',
 'soc.religion.christian',
 'talk.politics.guns',
 'talk.politics.mideast',
 'talk.politics.misc',
 'talk.religion.misc']

## Similaridad de documentos

Veamos similaridad de documentos. Tomemos algún documento

In [15]:
idx = 4811
print(newsgroups_train.data[idx])

THE WHITE HOUSE

                  Office of the Press Secretary
                   (Pittsburgh, Pennslyvania)
______________________________________________________________
For Immediate Release                         April 17, 1993     

             
                  RADIO ADDRESS TO THE NATION 
                        BY THE PRESIDENT
             
                Pittsburgh International Airport
                    Pittsburgh, Pennsylvania
             
             
10:06 A.M. EDT
             
             
             THE PRESIDENT:  Good morning.  My voice is coming to
you this morning through the facilities of the oldest radio
station in America, KDKA in Pittsburgh.  I'm visiting the city to
meet personally with citizens here to discuss my plans for jobs,
health care and the economy.  But I wanted first to do my weekly
broadcast with the American people. 
             
             I'm told this station first broadcast in 1920 when
it reported that year's presidential elec

Medimos la similaridad coseno con todos los documentos de train

In [16]:
cossim = cosine_similarity(X_train[idx], X_train)[0]

Podemos ver los valores de similaridad ordenados de mayor a menor

In [17]:
np.sort(cossim)[::-1]

array([1.        , 0.70930477, 0.67474953, ..., 0.        , 0.        ,
       0.        ], shape=(11314,))

Después vemos a qué documentos corresponden

In [18]:
np.argsort(cossim)[::-1]

array([4811, 6635, 4253, ..., 1911, 1825, 1828], shape=(11314,))

Obtenemos los 5 documentos más similares:

In [19]:
mostsim = np.argsort(cossim)[::-1][1:6]
print(mostsim)

[6635 4253 3596 4271 3746]


El documento original pertenece a la clase:

In [20]:
newsgroups_train.target_names[y_train[idx]]

'talk.politics.misc'

Revisamos las clases de los 5 más similares:

In [21]:
for i in mostsim:
  print(newsgroups_train.target_names[y_train[i]])

talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc


### Modelo de clasificación Naïve Bayes

Instanciamos el modelo de clasificación Naive Bayes y lo entrenamos con sklearn

In [22]:
clf = MultinomialNB()
clf.fit(X_train, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
Name,Type,Value
"class_count_ class_count_: ndarray of shape (n_classes,)Number of samples encountered for each class during fitting. Thisvalue is weighted by the sample weight when provided.","ndarray[float64](20,)","[480.,584.,591.,...,564.,465.,377.]"
"class_log_prior_ class_log_prior_: ndarray of shape (n_classes,)Smoothed empirical log probability for each class.","ndarray[float64](20,)","[-3.16,-2.96,-2.95,...,-3. ,-3.19,-3.4 ]"
"classes_ classes_: ndarray of shape (n_classes,)Class labels known to the classifier","ndarray[int64](20,)","[ 0, 1, 2,...,17,18,19]"
"feature_count_ feature_count_: ndarray of shape (n_classes, n_features)Number of samples encountered for each (class, feature)during fitting. This value is weighted by the sample weight whenprovided.","ndarray[float64](20, 101631)","[[0. ,0.94,0. ,...,0. ,0. ,0. ], [1.39,0.6 ,0. ,...,0. ,0. ,0. ], [0.95,0.14,0. ,...,0. ,0. ,0. ], ..., [0.42,2.9 ,0.04,...,0. ,0. ,0. ], [0.61,1.36,0. ,...,0. ,0. ,0. ], [0.03,0.38,0. ,...,0. ,0. ,0. ]]"
"feature_log_prob_ feature_log_prob_: ndarray of shape (n_classes, n_features)Empirical log probability of featuresgiven a class, ``P(x_i|y)``.","ndarray[float64](20, 101631)","[[-11.56,-10.9 ,-11.56,...,-11.56,-11.56,-11.56], [-10.69,-11.1 ,-11.56,...,-11.56,-11.56,-11.56], [-10.9 ,-11.44,-11.57,...,-11.57,-11.57,-11.57], ..., [-11.22,-10.21,-11.54,...,-11.57,-11.57,-11.57], [-11.09,-10.7 ,-11.56,...,-11.56,-11.56,-11.56], [-11.53,-11.23,-11.56,...,-11.56,-11.56,-11.56]]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,101631


Ya tenemos nuestro vectorizador ya ajustado en train, vectorizamos los textos
del conjunto de test.

In [23]:
X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target
y_pred =  clf.predict(X_test)

El F1-score es una métrica adecuada para evaluar el desempeño de modelos de clasificación, especialmente cuando existe desbalance entre clases.

* El promediado macro calcula el promedio del F1-score de cada clase, otorgando el mismo peso a todas las clases.
* El promediado micro calcula las métricas de forma global considerando todas las predicciones; en problemas de clasificación multiclase suele ser equivalente a la accuracy, por lo que no es la mejor métrica cuando el dataset está desbalanceado.

In [24]:
f1_score(y_test, y_pred, average='macro')

0.5854345727938506

---

## **Consigna del Desafío 1**
**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado.**



**1. Vectorizar documentos**
* Tomar 5 documentos al azar y medir similaridad con el resto de los documentos.
Estudiar los 5 documentos más similares de cada uno analizar si tiene sentido
la similaridad según el contenido del texto y la etiqueta de clasificación.

**2. Construir un modelo de clasificación por prototipos (tipo zero-shot).**
* Clasificar los documentos de un conjunto de test comparando cada uno con todos los de entrenamiento y asignar la clase al label del documento del conjunto de entrenamiento con mayor similaridad.

**3. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación**

* F1-Score Macro en el conjunto de datos de test. Considerar cambiar parámetros
de instanciación del vectorizador y los modelos y probar modelos de Naïve Bayes Multinomial y ComplementNB.

**NO cambiar el hiperparámetro ngram_range de los vectorizadores**.

**4. Transponer la matriz documento-término.**
* De esa manera se obtiene una matriz término-documento que puede ser interpretada como una colección de vectorización de palabras.
* Estudiar ahora similaridad entre palabras tomando 5 palabras y estudiando sus 5 más similares.

**Elegir las palabras MANUALMENTE para evitar la aparición de términos poco interpretables**.


In [25]:
np.random.seed(42)  # para reproducibilidad
indices_random = np.random.choice(X_train.shape[0], size=5, replace=False)
print(indices_random)

[7492 3546 5582 4793 3813]


In [26]:
for idx in indices_random:
    cossim = cosine_similarity(X_train[idx], X_train)[0]
    mostsim = np.argsort(cossim)[::-1][1:6]  # excluimos el propio documento (posición 0)
    
    print(f"=== Documento {idx} ===")
    print(f"Clase real: {newsgroups_train.target_names[y_train[idx]]}")
    print(f"Texto: {newsgroups_train.data[idx][:200]}")
    print(f"\nTop 5 similares:")
    for i in mostsim:
        print(f"  - Doc {i} | Similaridad: {cossim[i]:.3f} | Clase: {newsgroups_train.target_names[y_train[i]]}")
    print("\n" + "="*80 + "\n")

=== Documento 7492 ===
Clase real: comp.sys.mac.hardware
Texto: Could someone please post any info on these systems.

Thanks.
BoB
-- 
---------------------------------------------------------------------- 
Robert Novitskey | "Pursuing women is similar to banging o

Top 5 similares:
  - Doc 10935 | Similaridad: 0.667 | Clase: comp.sys.mac.hardware
  - Doc 7258 | Similaridad: 0.348 | Clase: comp.sys.ibm.pc.hardware
  - Doc 4971 | Similaridad: 0.180 | Clase: comp.sys.mac.hardware
  - Doc 4303 | Similaridad: 0.155 | Clase: misc.forsale
  - Doc 645 | Similaridad: 0.141 | Clase: comp.sys.mac.hardware


=== Documento 3546 ===
Clase real: comp.os.ms-windows.misc
Texto: 

     Don't bother if you have CPBackup or Fastback.  They all offer options 
not available in the stripped-down MS version (FROM CPS!).  Examples - no 
proprietary format (to save space), probably n

Top 5 similares:
  - Doc 5665 | Similaridad: 0.204 | Clase: comp.sys.ibm.pc.hardware
  - Doc 2011 | Similaridad: 0.192 | Clase: 

Observaciones: 
- Caso 1: la categoria del documento es "comp.sys.mac.hardware" y la categoria del documento mas similar es la misma. Y obtiene un valor alto de 0,67 lo cual es muy llamativo ya que ninguno de los otros 5 documentos obtiene un valor tan alto.
- Caso 2: esta vez, la categoria no concide con los top 5 similares, y en general los 5 documentos mas similares tampoco lo hacen, pero hay una particularidad y es que los 5 documentos mas similares tienen la misma categoria entre si, lo que nos puede estar diciendo que el documento elegido comparte muchas palabras con esa categoria, por eso salieron todos los similares con la misma categoria. Y como tambien hablan de tecnologia lo encuentro razonable.
- Caso 3: En este caso se obtiene un score de similaridad descente de 0,46 y ademas se acierta la categoria del documento original, lo cual es bueno, y tambien el 2do documento similar tambien lo hace. Considero que es un acierto por parte del algoritmo
- Caso 4: Para este caso, nuevamente tenemos un score muy bajo, 0,23 es bastante pobre, aunq por lo menos acierta la categoria, la particularidad es que obtiene scores muy cercano con los 5 mas similares, siendo el 2do el mismo score que el primero, aciertan la misma cantidad de palabras? siendo incluso de otra categoria, es almenos llamativo. Pero tambien por la categoria de documento que es lo encuentro razonable.
- Caso 5: Bajo score de similaridad con los 5 mas similares, y el mas similar ni siquiera acierta la cateogira, de hecho ninguno de los similares lo hace, es la primera vez que esto pasa, creo que es por la categoria del documento, quizas se me ocurre que no hay muchos documentos como este.

In [27]:
def clasificar_por_prototipo(X_test, X_train, y_train, batch_size=500):
    y_pred = []
    n_test = X_test.shape[0]
    
    for start in range(0, n_test, batch_size):
        end = min(start + batch_size, n_test)
        batch = X_test[start:end]
        sims = cosine_similarity(batch, X_train)  
        most_similar_idx = np.argmax(sims, axis=1)  # índice del más similar por fila
        y_pred.extend(y_train[most_similar_idx])
        print(f"Procesado {end}/{n_test}")
    
    return np.array(y_pred)

In [28]:
from sklearn.metrics import f1_score

y_pred_proto = clasificar_por_prototipo(X_test, X_train, y_train)
f1_proto = f1_score(y_test, y_pred_proto, average='macro')
print(f'F1-Score Macro (prototipos): {f1_proto:.4f}')

Procesado 500/7532
Procesado 1000/7532
Procesado 1500/7532
Procesado 2000/7532
Procesado 2500/7532
Procesado 3000/7532
Procesado 3500/7532
Procesado 4000/7532
Procesado 4500/7532
Procesado 5000/7532
Procesado 5500/7532
Procesado 6000/7532
Procesado 6500/7532
Procesado 7000/7532
Procesado 7500/7532
Procesado 7532/7532
F1-Score Macro (prototipos): 0.5050


In [29]:
# Experimento 1: baseline 
tfidf_base = TfidfVectorizer()
X_train_base = tfidf_base.fit_transform(newsgroups_train.data)
X_test_base = tfidf_base.transform(newsgroups_test.data)

clf_mnb = MultinomialNB()
clf_mnb.fit(X_train_base, y_train)
f1_mnb_base = f1_score(y_test, clf_mnb.predict(X_test_base), average='macro')
print(f'MultinomialNB baseline: {f1_mnb_base:.4f}')

clf_cnb = ComplementNB()
clf_cnb.fit(X_train_base, y_train)
f1_cnb_base = f1_score(y_test, clf_cnb.predict(X_test_base), average='macro')
print(f'ComplementNB baseline: {f1_cnb_base:.4f}')

MultinomialNB baseline: 0.5854
ComplementNB baseline: 0.6930


ComplimentNB esta diseñada para el desbalance de las clases, donde se que tengo clases con muy pocas palabras,  ya sea por pocos documentos o por documentos cortos. Esto nos da una estimacion un poco mejor que su competencia 

In [30]:
# Experimento 2: vectorizador con stopwords + filtrado de vocabulario
tfidf_v2 = TfidfVectorizer(stop_words='english', min_df=2, max_df=0.9)
X_train_v2 = tfidf_v2.fit_transform(newsgroups_train.data)
X_test_v2 = tfidf_v2.transform(newsgroups_test.data)

for alpha in [0.01, 0.1, 0.5, 1.0]:
    clf = ComplementNB(alpha=alpha)
    clf.fit(X_train_v2, y_train)
    f1 = f1_score(y_test, clf.predict(X_test_v2), average='macro')
    print(f'ComplementNB (alpha={alpha}): {f1:.4f}')

ComplementNB (alpha=0.01): 0.6729
ComplementNB (alpha=0.1): 0.6887
ComplementNB (alpha=0.5): 0.6974
ComplementNB (alpha=1.0): 0.6943



Con alpha muy bajo (0.01) el resultado empeora bastante, poco suavizado, hace que el modelo sea demasiado "confiado" en las frecuencias vistas en train, lo cual generaliza peor a documentos nuevos

Vemos que el valor optimo de alpha esta entre 0,5 y 1

In [31]:
tfidf_v3 = TfidfVectorizer(stop_words='english', min_df=2, max_df=0.9, sublinear_tf=True)
X_train_v3 = tfidf_v3.fit_transform(newsgroups_train.data)
X_test_v3 = tfidf_v3.transform(newsgroups_test.data)

for alpha in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7]:
    clf = ComplementNB(alpha=alpha)
    clf.fit(X_train_v3, y_train)
    f1 = f1_score(y_test, clf.predict(X_test_v3), average='macro')
    print(f'ComplementNB sublinear_tf (alpha={alpha}): {f1:.4f}')

ComplementNB sublinear_tf (alpha=0.2): 0.6932
ComplementNB sublinear_tf (alpha=0.3): 0.6930
ComplementNB sublinear_tf (alpha=0.4): 0.6942
ComplementNB sublinear_tf (alpha=0.5): 0.6951
ComplementNB sublinear_tf (alpha=0.6): 0.6951
ComplementNB sublinear_tf (alpha=0.7): 0.6946


sublinear_tf no ayudó en este caso en realidadempeoró levemente respecto a no usarlo (0.6951 vs 0.6974). Tiene sentido ya que sublinear_tf comprime el efecto de palabras repetidas dentro de un documento, pero en textos relativamente cortos como estos de newsgroups, esa repetición podria ser una señal útil de tema, y quizas por eso empeora el rendimiento. Al aplastar esa señal, se pierde algo de poder discriminativo.

Finalmente decido quedarme con mi mejor modelo que fue el anterior

In [32]:
X_train_T = X_train.T
print(f'Shape original: {X_train.shape}')
print(f'Shape transpuesta: {X_train_T.shape}')

Shape original: (11314, 101631)
Shape transpuesta: (101631, 11314)


In [34]:
palabras = ['orbit', 'god', 'computer', 'space', 'messi']

# Verificamos que existan en el vocabulario
for p in palabras:
    print(p, '->', tfidfvect.vocabulary_.get(p, 'NO ENCONTRADA'))

orbit -> 68433
god -> 43842
computer -> 28940
space -> 84097
messi -> 61083


In [36]:
X_train_T = X_train.T  # ahora cada fila es una palabra

palabras = ['orbit', 'god', 'computer', 'space', 'messi']

for palabra in palabras:
    idx_palabra = tfidfvect.vocabulary_[palabra]
    
    cossim = cosine_similarity(X_train_T[idx_palabra], X_train_T)[0]
    mostsim = np.argsort(cossim)[::-1][1:6]  # excluimos la propia palabra
    
    print(f"=== Palabra: '{palabra}' ===")
    for i in mostsim:
        print(f"  - {idx2word[i]} | Similaridad: {cossim[i]:.3f}")
    print()

=== Palabra: 'orbit' ===
  - hiten | Similaridad: 0.461
  - hagoromo | Similaridad: 0.367
  - lunar | Similaridad: 0.352
  - subsatellite | Similaridad: 0.341
  - flybys | Similaridad: 0.336

=== Palabra: 'god' ===
  - jesus | Similaridad: 0.269
  - bible | Similaridad: 0.262
  - that | Similaridad: 0.256
  - existence | Similaridad: 0.255
  - christ | Similaridad: 0.251

=== Palabra: 'computer' ===
  - decwriter | Similaridad: 0.156
  - deluged | Similaridad: 0.152
  - harkens | Similaridad: 0.152
  - shopper | Similaridad: 0.144
  - the | Similaridad: 0.136

=== Palabra: 'space' ===
  - nasa | Similaridad: 0.330
  - seds | Similaridad: 0.297
  - shuttle | Similaridad: 0.293
  - enfant | Similaridad: 0.280
  - seti | Similaridad: 0.246

=== Palabra: 'messi' ===
  - messi | Similaridad: 1.000
  - julkunen | Similaridad: 1.000
  - 734086202 | Similaridad: 1.000
  - pharmaceutical | Similaridad: 0.999
  - antero | Similaridad: 0.753



Caso 1: palabras muy relacionadas con orbit, busque en internet y "Hiten" y "hagoromo" son nombres de sondas espaciales japonesas reales de esa época. Asi q fueron bien encontradas.
Caso 2: Palabras tambien totalmente relacionadas salvo por "that" que es una palabra de gramatica (creo) en ingles, aparece pq no use stopwords, lo cual deja que entren estas palabras en "forma de ruido" y de hecho saco mas puntaje que "existence"
Caso 3: Aca falla, ya que las palabras relacionadas tienen puntuacion muy baja y de por si las palabras no tienen relacion "tematica" muy clara, y eso que computer es una palabra no tan generica. Tambien aparece "the" como palabra parecida
Caso 4: space obtiene buena nota en nasa y demas palabras salvo por "enfant" que aparece ahi no se por que, es facil caer en que es un fallo de similaridad pero si pienso que quizas algun texto de niños diga que les gustan los astronautas y tal, creo que puede venir por ahi y tiene sentido para mi.
Caso 5: elegi messi pq era una palabra muy extraña para ver como se comportaba, y bueno de hecho olvide que el dataset era de los 90, Messi ni jugaba en esa epoca, encima logra similaridad 1 con 3 palabras, se me ocurre pq son palabras que aparecen 1 sola vez y por eso obtienen similaridad tan alta, pero hace que el algoritmo falle, deberia dar 0 con todas. De todas formas fue una buena palabra para ver un caso extremo para el algoritmo


Conclusion:Palabras frecuentes y temáticamente específicas como orbit, space y god dan vecinos semánticamente coherentes. Palabras muy genéricas como computer dan similaridades débiles y ruidosas. Palabras muy raras messi producen similitudes artificialmente perfectas por coincidencia estadística